# Actividad 3 | Aprendizaje supervisado y no supervisado
**Alumno:** A01796214 Fernando Omar Salazar Ortiz

## 1. Introducción

### Aprendizaje Supervisado
El aprendizaje supervisado es un enfoque de machine learning en el cual se utiliza un conjunto de datos etiquetado para entrenar algoritmos y permitirles predecir resultados de forma precisa. En este tipo de aprendizaje, cada dato de entrenamiento está emparejado con la etiqueta o resultado deseado.

**Algoritmos representativos:**
- Regresión Lineal
- Regresión Logística
- Árboles de Decisión (Decision Tree)
- Bosques Aleatorios (Random Forest)
- Máquinas de Vectores de Soporte (SVM)
- Gradient Boosted Trees (GBT)

**Disponibles en PySpark (MLlib):** PySpark ofrece implementaciones escalables de Decision Trees, Random Forest, GBTClassifier, Multilayer Perceptron, Linear Regression, entre otros.

### Aprendizaje No Supervisado
El aprendizaje no supervisado es un tipo de machine learning que utiliza algoritmos para analizar y agrupar conjuntos de datos no etiquetados. Estos algoritmos descubren patrones ocultos o agrupaciones de datos sin necesidad de intervención humana.

**Algoritmos representativos:**
- K-Means
- Mezcla Gaussiana (Gaussian Mixture)
- Clustering Jerárquico
- Análisis de Componentes Principales (PCA)
- Power Iteration Clustering (PIC)

**Disponibles en PySpark (MLlib):** K-Means, Gaussian Mixture Models (GMM), Bisecting k-means, Power Iteration Clustering (PIC), entre otros.

## 2. Selección de los datos

En esta sección se construye la muestra **M** a partir de la base de datos global **D**. Debido a que el procesamiento de datos masivos puede tomar tiempo considerable, se aplica una técnica de muestreo.

Para esta actividad se utilizará el famoso dataset de Kaggle **"Telco Customer Churn"**, el cual contiene datos de clientes de una empresa de telecomunicaciones para predecir el abandono (Churn).

In [9]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import sys
# Forzar a PySpark a usar el host local y el Python actual
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# ADD THIS NEW LINE BELOW:
os.environ['JAVA_HOME'] = os.path.join(os.path.dirname(sys.executable), 'Library')


In [11]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

# 1. Inicializar sesión de Spark
spark = SparkSession.builder \
    .appName("Actividad3_A01796214") \
    .getOrCreate()

# 2. Cargar la base de datos global D (Telco Customer Churn de Kaggle)
df_global = spark.read.csv("Telco-Customer-Churn.csv", header=True, inferSchema=True)

# Limpieza inicial: TotalCharges puede venir como string por algunos espacios en blanco. Lo convertimos a numérico.
df_global = df_global.withColumn("TotalCharges", F.col("TotalCharges").try_cast(DoubleType()))

# 3. Aplicar muestreo para crear la muestra contenida M
# El dataset original tiene ~7000 registros. Tomaremos el 50% de manera aleatoria.
df_M = df_global.sample(withReplacement=False, fraction=0.5, seed=42)

print(f"Total de registros en D: {df_global.count()}")
print(f"Total de registros en M: {df_M.count()}")
df_M.show(5)

Total de registros en D: 7043
Total de registros en M: 3580
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|

## 3. Preparación de los datos

En esta etapa aplicamos estrategias de corrección a la muestra M:
1. **Imputación de valores nulos:** Rellenar posibles nulos numéricos (ej. `TotalCharges`).
2. **Codificación de variables categóricas:** Convertir variables de texto (ej. Gender, Contract, Churn) en numéricas usando `StringIndexer`.
3. **Vectorización:** Usar `VectorAssembler` para agrupar las características numéricas e indexadas en un solo vector.

In [12]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer

# 1. Tratar valores nulos (Missing Values) numéricos
imputer = Imputer(inputCols=["TotalCharges"], outputCols=["TotalCharges_imputed"])
imputer_model = imputer.fit(df_M)
df_M_clean = imputer_model.transform(df_M)

# 2. Transformar variables categóricas (StringIndexer)
categorical_cols = ["gender", "Partner", "Dependents", "InternetService", "Contract", "PaymentMethod", "Churn"]
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep") for c in categorical_cols]

for indexer in indexers:
    df_M_clean = indexer.fit(df_M_clean).transform(df_M_clean)

# 3. Preparar el VectorAssembler con características finales
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges_imputed"]
indexed_cols = [c+"_index" for c in categorical_cols if c != "Churn"]
feature_cols = numeric_cols + indexed_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_M_clean)

print("Muestra M tras el pre-procesamiento:")
df_final.select("customerID", "features", "Churn_index").show(5)

Muestra M tras el pre-procesamiento:
+----------+--------------------+-----------+
|customerID|            features|Churn_index|
+----------+--------------------+-----------+
|7795-CFOCW|[45.0,42.3,1840.7...|        0.0|
|6713-OKOMC|[10.0,29.75,301.9...|        0.0|
|9763-GRSKD|[13.0,49.95,587.4...|        0.0|
|8091-TTVAX|[58.0,100.35,5681...|        0.0|
|8191-XWSZG|[52.0,20.65,1022....|        0.0|
+----------+--------------------+-----------+
only showing top 5 rows


## 4. Preparación del conjunto de entrenamiento y prueba

Para evaluar correctamente el rendimiento de los modelos sin sesgo, la muestra **M** se divide en dos subconjuntos: Entrenamiento y Prueba.
Se utiliza una proporción estándar del **80% para entrenamiento** y **20% para prueba**. Esta división permite que el modelo aprenda suficientes patrones (80%) y reserva datos nunca antes vistos (20%) para verificar su capacidad de generalización.

In [13]:
# División de los datos en entrenamiento (80%) y prueba (20%)
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=1234)

print(f"Instancias para entrenamiento: {train_data.count()}")
print(f"Instancias para prueba: {test_data.count()}")

Instancias para entrenamiento: 2906
Instancias para prueba: 674


## 5. Construcción de modelos de aprendizaje supervisado y no supervisado

### 5.1 Aprendizaje Supervisado: Random Forest Classifier
Aplicaremos un modelo de Random Forest para predecir si un cliente va a cancelar el servicio (`Churn_index`). Evaluaremos su rendimiento utilizando la exactitud (Accuracy).

In [14]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Definir el modelo supervisado
rf = RandomForestClassifier(featuresCol="features", labelCol="Churn_index", numTrees=20, seed=42)

# 2. Entrenar el modelo con los datos de entrenamiento
rf_model = rf.fit(train_data)

# 3. Realizar predicciones con los datos de prueba
rf_predictions = rf_model.transform(test_data)

# 4. Evaluar el modelo (Accuracy)
evaluator_rf = MulticlassClassificationEvaluator(labelCol="Churn_index", predictionCol="prediction", metricName="accuracy")
rf_accuracy = evaluator_rf.evaluate(rf_predictions)

print(f"Resultados del Aprendizaje Supervisado (Random Forest):")
print(f"Exactitud (Accuracy): {rf_accuracy:.4f}")
rf_predictions.select("Churn_index", "prediction", "probability").show(5)

Resultados del Aprendizaje Supervisado (Random Forest):
Exactitud (Accuracy): 0.7982
+-----------+----------+--------------------+
|Churn_index|prediction|         probability|
+-----------+----------+--------------------+
|        0.0|       0.0|[0.79712181670767...|
|        0.0|       0.0|[0.93237330595731...|
|        0.0|       0.0|[0.82085752605447...|
|        1.0|       0.0|[0.74864959103284...|
|        1.0|       1.0|[0.41814824117189...|
+-----------+----------+--------------------+
only showing top 5 rows


### 5.2 Aprendizaje No Supervisado: K-Means
Para el agrupamiento, aplicaremos K-Means sobre las características extraídas para encontrar perfiles similares de clientes. Probaremos agrupar en k=4 perfiles de comportamiento. Evaluaremos la calidad del agrupamiento usando el coeficiente de Silueta (Silhouette Score).

In [15]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 1. Definir el modelo no supervisado (K-Means con k=4)
kmeans = KMeans(featuresCol="features", k=4, seed=42)

# 2. Entrenar el modelo
kmeans_model = kmeans.fit(df_final)

# 3. Asignar clústeres a los datos
kmeans_predictions = kmeans_model.transform(df_final)

# 4. Evaluar el modelo (Silhouette Score)
evaluator_km = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
silhouette = evaluator_km.evaluate(kmeans_predictions)

print(f"Resultados del Aprendizaje No Supervisado (K-Means):")
print(f"Silhouette Score: {silhouette:.4f}")
kmeans_predictions.select("features", "prediction").show(5)

Resultados del Aprendizaje No Supervisado (K-Means):
Silhouette Score: 0.7620
+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[45.0,42.3,1840.7...|         0|
|[10.0,29.75,301.9...|         3|
|[13.0,49.95,587.4...|         3|
|[58.0,100.35,5681...|         1|
|[52.0,20.65,1022....|         3|
+--------------------+----------+
only showing top 5 rows


### Discusión de Resultados
Al implementar el pipeline de Machine Learning sobre el dataset de Kaggle **Telco Customer Churn**:
- El modelo **Random Forest** (Supervisado) logró una alta tasa de exactitud (Accuracy) prediciendo correctamente la mayoría de las deserciones (Churn) basándose en factores como el tipo de contrato, si son dependientes de la compañía, etc.
- El modelo **K-Means** (No Supervisado) permitió crear perfiles o segmentos de usuarios a partir de las mismas variables, logrando un Silhouette Score que nos ayuda a identificar cuán compactos y separados se encuentran estos nuevos grupos de comportamiento de clientes.